# PGVector functional walkthrough

This notebook mirrors the PGVector functional-test flow defined by:

- `tests/functional/conftest.py`
- `tests/functional/data.py`
- `tests/functional/filter_assertions.py`
- `tests/functional/test_pgvector_filters.py`

It is designed for engineers who want to bring up the PGVector-backed stack locally,
seed the deterministic functional-test fixtures, and manually inspect how
`vector-retriever` behaves across health, readiness, capabilities, batch querying,
explainable filters, top-k limiting, and the full filter matrix.


## Prerequisites

Before running the notebook:

- start from a checkout of this repository
- make sure Docker Engine and the `docker compose` plugin are available
- ensure ports `6103`, `9713`, and `5433` are free on the host
- expect the first startup to take longer if images or models need to be pulled/downloaded
- run the notebook top-to-bottom for the cleanest experience

The notebook is self-contained: all runtime helpers live in notebook code cells, while
fixture constants are imported directly from the functional-test source of truth.


In [1]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from copy import deepcopy
from pathlib import Path
from typing import Any
from uuid import uuid4

import requests
from IPython.display import JSON, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / "pyproject.toml").exists() and (path / "docker" / "compose.yaml").exists():
            return path
    raise RuntimeError(
        "Could not locate the vector-retriever repository root from the current working directory."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tests.functional.data import DOCUMENTS, FILTER_CASES  # noqa: E402

display(Markdown(f"**Repository root:** `{REPO_ROOT}`"))
display(Markdown(f"**Fixture documents imported:** `{len(DOCUMENTS)}`"))
display(Markdown(f"**Filter cases imported:** `{len(FILTER_CASES)}`"))


**Repository root:** `/home/intel/workbench/integration/lib/microservices/vector-retriever/vector-retriever`

**Fixture documents imported:** `6`

**Filter cases imported:** `20`

## PGVector-specific environment and ports

This section defines the PGVector backend settings used by the functional suite:

- `VECTOR_RETRIEVER_HOST_PORT=6103`
- `EMBEDDING_SERVER_PORT=9713`
- `PGVECTOR_HOST_PORT=5433`

It also creates a unique index name for the walkthrough so repeated runs do not collide.


In [2]:
BACKEND = "pgvector"
PORT_MAP = {
    "pgvector": {
        "VECTOR_RETRIEVER_HOST_PORT": "6103",
        "EMBEDDING_SERVER_PORT": "9713",
        "PGVECTOR_HOST_PORT": "5433",
    }
}
COMPOSE_PROJECT = "docker"
OV_MODELS_SOURCE_VOLUME = "docker_ov_models"
OV_MODELS_DEST_VOLUME = f"{COMPOSE_PROJECT}_ov-models"
DEFAULT_TEST_EMBEDDING_MODEL_NAME = "CLIP/clip-vit-b-32"
COMPOSE_FILES = ["docker/compose.yaml", f"docker/compose.{BACKEND}.yaml"]


def build_env(backend: str = BACKEND) -> dict[str, str]:
    env = os.environ.copy()
    env.update(PORT_MAP[backend])
    env["RETRIEVER_BACKEND"] = backend
    index_name = f"vr_functional_{backend}_{uuid4().hex[:8]}"
    env["INDEX_NAME"] = index_name
    env["VS_INDEX_NAME"] = index_name
    env["EMBEDDING_MODEL_NAME"] = (
        env.get("EMBEDDING_MODEL_NAME", "").strip() or DEFAULT_TEST_EMBEDDING_MODEL_NAME
    )
    env["EMBEDDING_DEVICE"] = "CPU"
    env["EMBEDDING_USE_OV"] = "true"
    env["VECTOR_RETRIEVER_LOG_LEVEL"] = "debug"
    return env


SESSION: dict[str, Any] = {
    "backend": BACKEND,
    "env": build_env(),
    "compose_files": COMPOSE_FILES,
    "started": False,
    "seeded": False,
}
SESSION["base_url"] = f"http://localhost:{SESSION['env']['VECTOR_RETRIEVER_HOST_PORT']}"

display(
    JSON(
        {
            "backend": SESSION["backend"],
            "base_url": SESSION["base_url"],
            "ports": PORT_MAP[BACKEND],
            "index_name": SESSION["env"]["INDEX_NAME"],
            "embedding_model": SESSION["env"]["EMBEDDING_MODEL_NAME"],
            "compose_files": SESSION["compose_files"],
        },
        expanded=True,
    )
)


<IPython.core.display.JSON object>

## Notebook-local helper functions

These helpers intentionally mirror the functional test flow without importing test helpers:

- Docker Compose execution for the PGVector overlay
- optional OpenVINO model volume warm-up
- readiness polling
- deterministic seeding into the active backend
- HTTP helpers for `/health`, `/ready`, `/capabilities/filters`, and `/query`
- compact result rendering for manual inspection
- teardown utilities


In [3]:
def display_json(payload: Any) -> None:
    display(JSON(payload, expanded=True))


def ensure_docker_available() -> None:
    if shutil.which("docker") is None:
        raise RuntimeError("docker CLI is required to run this walkthrough")


def compose_base_command(session: dict[str, Any]) -> list[str]:
    command = ["docker", "compose"]
    for compose_file in session["compose_files"]:
        command.extend(["-f", compose_file])
    return command


def run_compose(session: dict[str, Any], args: list[str]) -> subprocess.CompletedProcess[str]:
    command = [*compose_base_command(session), *args]
    print("$", " ".join(command))
    try:
        completed = subprocess.run(
            command,
            cwd=REPO_ROOT,
            env=session["env"],
            check=True,
            text=True,
            capture_output=True,
        )
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f"docker compose command failed (exit {exc.returncode}):\n"
            f"cmd: {' '.join(exc.cmd)}\n"
            f"stdout:\n{exc.stdout}\n"
            f"stderr:\n{exc.stderr}"
        ) from exc
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return completed


def docker_cli(command: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        command,
        cwd=REPO_ROOT,
        check=check,
        text=True,
        capture_output=True,
    )


def ensure_ov_models_volume() -> None:
    if docker_cli(["docker", "volume", "inspect", OV_MODELS_DEST_VOLUME], check=False).returncode == 0:
        return
    if docker_cli(["docker", "volume", "inspect", OV_MODELS_SOURCE_VOLUME], check=False).returncode != 0:
        return
    docker_cli(["docker", "volume", "create", OV_MODELS_DEST_VOLUME])
    docker_cli(
        [
            "docker",
            "run",
            "--rm",
            "-v",
            f"{OV_MODELS_SOURCE_VOLUME}:/src",
            "-v",
            f"{OV_MODELS_DEST_VOLUME}:/dst",
            "alpine",
            "sh",
            "-c",
            "cp -r /src/. /dst/",
        ]
    )


def compose_ps(session: dict[str, Any]) -> str:
    return run_compose(session, ["ps"]).stdout.strip()


def start_services(session: dict[str, Any]) -> str:
    ensure_docker_available()
    ensure_ov_models_volume()
    run_compose(session, ["up", "-d", "--build"])
    session["started"] = True
    return compose_ps(session)


def teardown_stack(session: dict[str, Any], *, remove_volumes: bool = True) -> str:
    args = ["down"]
    if remove_volumes:
        args.append("-v")
    args.append("--remove-orphans")
    result = run_compose(session, args)
    session["started"] = False
    session["seeded"] = False
    return result.stdout.strip()


def wait_for_ready(base_url: str, timeout_seconds: int = 600) -> dict[str, Any]:
    start = time.time()
    last_error = ""
    while time.time() - start < timeout_seconds:
        try:
            response = requests.get(f"{base_url}/ready", timeout=10)
            if response.status_code == 200:
                payload = response.json()
                if payload.get("status") == "ready":
                    return payload
                last_error = f"unexpected /ready payload: {payload}"
            else:
                last_error = f"/ready status={response.status_code} body={response.text}"
        except Exception as exc:  # noqa: BLE001
            last_error = str(exc)
        time.sleep(3)
    raise AssertionError(f"vector-retriever did not become ready: {last_error}")


def api_get(session: dict[str, Any], path: str, *, timeout: int = 15) -> dict[str, Any]:
    response = requests.get(f"{session['base_url']}{path}", timeout=timeout)
    response.raise_for_status()
    return response.json()


def query_api(
    session: dict[str, Any],
    payload: list[dict[str, Any]],
    *,
    timeout: int = 60,
    allow_errors: bool = False,
) -> dict[str, Any]:
    response = requests.post(f"{session['base_url']}/query", json=payload, timeout=timeout)
    response.raise_for_status()
    body = response.json()
    if not allow_errors:
        assert not body["errors"], body["errors"]
    return body


def seed_backend_data(session: dict[str, Any]) -> str:
    docs_literal = json.dumps(DOCUMENTS)
    script = (
        "import json\n"
        "from src.common.settings import settings\n"
        "from src.retriever.backend_factory import get_vectordb\n"
        f"expected_backend = {session['backend']!r}\n"
        "assert settings.RETRIEVER_BACKEND == expected_backend, (\n"
        "    f'backend mismatch: expected {expected_backend!r}, got {settings.RETRIEVER_BACKEND!r}'\n"
        ")\n"
        f"docs = json.loads({docs_literal!r})\n"
        "store = get_vectordb()\n"
        "texts = [item['page_content'] for item in docs]\n"
        "metadatas = [item['metadata'] for item in docs]\n"
        "ids = [item['metadata']['video_id'] for item in docs]\n"
        "try:\n"
        "    store.add_texts(texts=texts, metadatas=metadatas, ids=ids)\n"
        "except TypeError:\n"
        "    store.add_texts(texts=texts, metadatas=metadatas)\n"
        "print('seeded', len(docs))\n"
    )
    result = run_compose(session, ["exec", "-T", "vector-retriever", "python", "-c", script])
    session["seeded"] = True
    return result.stdout.strip()


def verify_seed_visible(session: dict[str, Any], expected_count: int) -> dict[str, Any]:
    payload = [
        {
            "query_id": "seed-verification",
            "query": "fixture retrieval anchor",
            "top_k": 100,
        }
    ]
    body = query_api(session, payload)
    result = body["results"][0]
    assert result["count"] >= expected_count, (
        f"expected at least {expected_count} seeded docs, got {result['count']}"
    )
    return result


def simplify_item(item: dict[str, Any]) -> dict[str, Any]:
    metadata = item["metadata"]
    return {
        "video_id": metadata.get("video_id"),
        "score": round(float(item.get("score", 0.0)), 6),
        "camera_zone": metadata.get("camera_zone"),
        "frame_number": metadata.get("frame_number"),
        "bucket_name": metadata.get("bucket_name"),
        "tags": metadata.get("tags"),
        "created_at": metadata.get("created_at"),
    }


def show_result_block(result: dict[str, Any]) -> None:
    display_json(
        {
            "query_id": result["query_id"],
            "query": result["query"],
            "count": result["count"],
            "applied_filters": result["applied_filters"],
            "items": [simplify_item(item) for item in result["items"]],
        }
    )


def show_batch_response(body: dict[str, Any]) -> None:
    display_json(
        {
            "request_id": body["request_id"],
            "errors": body["errors"],
            "results": [
                {
                    "query_id": result["query_id"],
                    "count": result["count"],
                    "items": [simplify_item(item) for item in result["items"]],
                    "applied_filters": result["applied_filters"],
                }
                for result in body["results"]
            ],
        }
    )


def assert_expected_ids(result: dict[str, Any], expected_ids: set[str], case_name: str) -> None:
    matched_ids = {item["metadata"]["video_id"] for item in result["items"]}
    assert matched_ids == expected_ids, (
        f"case={case_name} expected={sorted(expected_ids)} got={sorted(matched_ids)}"
    )


FILTER_CASES_BY_NAME = {case["name"]: case for case in FILTER_CASES}


def make_case_query(case: dict[str, Any], *, top_k: int = 100, explain_filters: bool = True) -> dict[str, Any]:
    payload = {
        "query_id": case["name"],
        "query": "fixture retrieval anchor",
        "top_k": top_k,
        "explain_filters": explain_filters,
    }
    payload.update(deepcopy(case["payload"]))
    return payload


def run_filter_case(session: dict[str, Any], case_name: str) -> dict[str, Any]:
    case = FILTER_CASES_BY_NAME[case_name]
    body = query_api(session, [make_case_query(case)])
    result = body["results"][0]
    assert_expected_ids(result, case["expected_ids"], case_name)
    show_result_block(result)
    return result


## Preview the deterministic fixture set

The same six fixture documents and filter cases used by the functional tests are imported
here so the walkthrough operates on exactly the same seed data and expectations.


In [4]:
fixture_preview = [
    {
        "video_id": doc["metadata"]["video_id"],
        "camera_zone": doc["metadata"]["camera_zone"],
        "frame_number": doc["metadata"]["frame_number"],
        "bucket_name": doc["metadata"]["bucket_name"],
        "tags": doc["metadata"]["tags"],
        "created_at": doc["metadata"]["created_at"],
        "optional_note": doc["metadata"].get("optional_note"),
    }
    for doc in DOCUMENTS
]

display_json(
    {
        "documents": fixture_preview,
        "filter_cases": [case["name"] for case in FILTER_CASES],
    }
)


<IPython.core.display.JSON object>

## Start the PGVector stack with Docker Compose

This cell brings up the same compose files used by the functional test suite:

- `docker/compose.yaml`
- `docker/compose.pgvector.yaml`

The output shows the current compose state after startup.


In [5]:
start_output = start_services(SESSION)
print(start_output)


$ docker compose -f docker/compose.yaml -f docker/compose.pgvector.yaml up -d --build


#1 [internal] load local bake definitions
#1 reading from stdin 940B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 1.10kB done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.12.13-slim
#3 DONE 0.9s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [1/8] FROM docker.io/library/python:3.12.13-slim@sha256:090ba77e2958f6af52a5341f788b50b032dd4ca28377d2893dcf1ecbdfdfe203
#5 resolve docker.io/library/python:3.12.13-slim@sha256:090ba77e2958f6af52a5341f788b50b032dd4ca28377d2893dcf1ecbdfdfe203 0.0s done
#5 CACHED

#6 [internal] load build context
#6 transferring context: 4.01kB done
#6 DONE 0.0s

#7 [2/8] RUN groupadd --gid 1000 appuser &&     useradd --uid 1000 --gid appuser --shell /bin/bash --create-home appuser
#7 DONE 0.2s

#8 [3/8] WORKDIR /app
#8 DONE 0.0s

#9 [4/8] COPY pyproject.toml README.md ./
#9 DONE 0.0s

#10 [5/8] RUN pip install --upgrade pip &&     pip install poetry

## Wait for readiness

The functional tests poll `/ready` until the service reports `status=ready`.
This cell mirrors that behavior and returns the readiness payload once the stack is live.


In [6]:
ready_payload = wait_for_ready(SESSION["base_url"])
display_json(ready_payload)


<IPython.core.display.JSON object>

## Seed deterministic fixture documents into PGVector

The notebook seeds the imported fixture documents by executing Python inside the running
`vector-retriever` container, matching the backend seeding flow from `tests/functional/conftest.py`.


In [7]:
seed_output = seed_backend_data(SESSION)
print(seed_output)


$ docker compose -f docker/compose.yaml -f docker/compose.pgvector.yaml exec -T vector-retriever python -c import json
from src.common.settings import settings
from src.retriever.backend_factory import get_vectordb
expected_backend = 'pgvector'
assert settings.RETRIEVER_BACKEND == expected_backend, (
    f'backend mismatch: expected {expected_backend!r}, got {settings.RETRIEVER_BACKEND!r}'
)
docs = json.loads('[{"page_content": "red car at intersection", "metadata": {"video_id": "vid-001", "camera_zone": "north", "description": "red car at intersection", "prefix": "alpha-route", "frame_number": 10, "bucket_name": "cam-a", "tags": ["traffic", "vehicle", "red"], "created_at": "2026-03-10T10:00:00+00:00", "optional_note": "doc has note"}}, {"page_content": "blue bus near station", "metadata": {"video_id": "vid-002", "camera_zone": "south", "description": "blue bus near station", "prefix": "beta-route", "frame_number": 20, "bucket_name": "cam-b", "tags": ["traffic", "vehicle", "bus"], "cre

## Verify the seed is visible through `/query`

Before exploring filters, confirm that the seeded fixture set is queryable through the live
service. This mirrors `_verify_seed_visible(...)` in the functional test fixtures.


In [8]:
seed_result = verify_seed_visible(SESSION, expected_count=len(DOCUMENTS))
show_result_block(seed_result)


<IPython.core.display.JSON object>

## Endpoint check: `/health`

`/health` is a liveness check. It should return `status="ok"` with a timestamp.


In [9]:
health_payload = api_get(SESSION, "/health")
assert health_payload["status"] == "ok", health_payload
display_json(health_payload)


<IPython.core.display.JSON object>

## Endpoint check: `/ready`

`/ready` confirms that the runtime dependencies are available and the active backend is ready
to serve queries.


In [10]:
ready_check_payload = api_get(SESSION, "/ready")
assert ready_check_payload["status"] == "ready", ready_check_payload
display_json(ready_check_payload)


<IPython.core.display.JSON object>

## Endpoint check: `/capabilities/filters`

This endpoint reports the active backend and the filter grammar capabilities exposed by each
supported backend. The active backend should be `pgvector`.


In [11]:
capabilities_payload = api_get(SESSION, "/capabilities/filters")
assert capabilities_payload["active_backend"] == SESSION["backend"], capabilities_payload
supported_backends = {backend["backend"] for backend in capabilities_payload["backends"]}
assert SESSION["backend"] in supported_backends, supported_backends
display_json(capabilities_payload)


<IPython.core.display.JSON object>

## Endpoint check: `/query` smoke test

This is a simple single-query request that filters by `video_id=vid-001` so the notebook can
confirm a precise PGVector lookup before moving on to batch and filter-matrix scenarios.


In [12]:
smoke_payload = [
    {
        "query_id": "query-smoke",
        "query": "fixture retrieval anchor",
        "where": {"field": "video_id", "op": "eq", "value": "vid-001"},
        "top_k": 5,
    }
]
smoke_body = query_api(SESSION, smoke_payload)
smoke_result = smoke_body["results"][0]
assert_expected_ids(smoke_result, {"vid-001"}, "query-smoke")
show_batch_response(smoke_body)


<IPython.core.display.JSON object>

## Batch query

The functional tests send two queries in a single `/query` request and expect each result set
to contain exactly one document.


In [13]:
batch_payload = [
    {
        "query_id": "batch-q1",
        "query": "fixture retrieval anchor",
        "where": {"field": "video_id", "op": "eq", "value": "vid-001"},
        "top_k": 10,
    },
    {
        "query_id": "batch-q2",
        "query": "fixture retrieval anchor",
        "where": {"field": "video_id", "op": "eq", "value": "vid-002"},
        "top_k": 10,
    },
]
batch_body = query_api(SESSION, batch_payload)
show_batch_response(batch_body)

ids_q1 = {item["metadata"]["video_id"] for item in batch_body["results"][0]["items"]}
ids_q2 = {item["metadata"]["video_id"] for item in batch_body["results"][1]["items"]}
assert ids_q1 == {"vid-001"}, ids_q1
assert ids_q2 == {"vid-002"}, ids_q2


<IPython.core.display.JSON object>

## `explain_filters=True`

This query requests filter explanation metadata. The functional expectation is that the
response includes a non-null `compiled_backend_filter` for the active backend.


In [14]:
explain_payload = [
    {
        "query_id": "explain-test",
        "query": "fixture retrieval anchor",
        "where": {"field": "frame_number", "op": "gte", "value": 50},
        "top_k": 10,
        "explain_filters": True,
    }
]
explain_body = query_api(SESSION, explain_payload)
explain_result = explain_body["results"][0]
compiled_backend_filter = explain_result["applied_filters"].get("compiled_backend_filter")
assert compiled_backend_filter is not None, explain_result["applied_filters"]
show_result_block(explain_result)
display_json({"compiled_backend_filter": compiled_backend_filter})


<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

## `top_k` limiting

The functional tests confirm that `top_k=2` truncates the result set to exactly two returned
items when more matches are available.


In [15]:
topk_payload = [
    {
        "query_id": "topk-limit",
        "query": "fixture retrieval anchor",
        "top_k": 2,
    }
]
topk_body = query_api(SESSION, topk_payload)
topk_result = topk_body["results"][0]
assert topk_result["count"] == 2, topk_result["count"]
assert len(topk_result["items"]) == 2, len(topk_result["items"])
show_result_block(topk_result)


<IPython.core.display.JSON object>

## Image query

This demonstrates the image query modality introduced alongside the existing text query.
An image can be provided as `image_base64` or `image_url` instead of a text `query`.
The two modalities are mutually exclusive — providing both returns a `422` validation error.

The test uses a tiny 1×1 red PNG encoded in base64 so it works without network access.

In [16]:
RED_1X1_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR4nGP4"
    "z8BQDwAEgAF/pooBPQAAAABJRU5ErkJggg=="
)

# --- image_base64 query ---
image_payload = [
    {
        "query_id": "image-base64-test",
        "image": {"type": "image_base64", "image_base64": RED_1X1_PNG_B64},
        "top_k": 5,
    }
]
image_body = query_api(SESSION, image_payload)
image_result = image_body["results"][0]
assert image_result["query_id"] == "image-base64-test"
assert image_result["query"] == "[image_base64]", image_result["query"]
assert image_result["count"] >= 1, image_result["count"]
show_result_block(image_result)

# --- image query with where filter ---
filtered_payload = [
    {
        "query_id": "image-filtered-test",
        "image": {"type": "image_base64", "image_base64": RED_1X1_PNG_B64},
        "where": {"field": "video_id", "op": "eq", "value": "vid-001"},
        "top_k": 10,
    }
]
filtered_body = query_api(SESSION, filtered_payload)
filtered_result = filtered_body["results"][0]
matched_ids = {item["metadata"]["video_id"] for item in filtered_result["items"]}
assert matched_ids <= {"vid-001"}, matched_ids
show_result_block(filtered_result)

# --- mutual exclusivity: both query and image should fail ---
invalid_response = requests.post(
    f"{SESSION['base_url']}/query",
    json=[{"query": "test", "image": {"type": "image_base64", "image_base64": RED_1X1_PNG_B64}}],
    timeout=60,
)
assert invalid_response.status_code == 422, invalid_response.status_code
print(f"Mutual exclusivity correctly rejected with {invalid_response.status_code}")


<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

Mutual exclusivity correctly rejected with 422


## Filter case: `op_eq`

            **Family:** Scalar equality

            This cell mirrors the `op_eq` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-001`.

            ```json
            {
  "where": {
    "field": "video_id",
    "op": "eq",
    "value": "vid-001"
  }
}
            ```


In [17]:
case_result = run_filter_case(SESSION, 'op_eq')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_eq',
 'query': 'fixture retrieval anchor',
 'count': 1,
 'items': [{'score': 0.7388502579584794,
   'metadata': {'tags': ['traffic', 'vehicle', 'red'],
    'prefix': 'alpha-route',
    'video_id': 'vid-001',
    'created_at': '2026-03-10T10:00:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'frame_number': 10,
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'field': 'video_id',
   'op': 'eq',
   'value': 'vid-001',
   'all': None,
   'any': None,
   'not': None},
  'warnings': None,
  'compiled_backend_filter': {'video_id': {'$eq': 'vid-001'}},
  'dropped_or_rewritten_clauses': []}}

## Filter case: `op_in`

            **Family:** Set membership

            This cell mirrors the `op_in` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-001`, `vid-003`.

            ```json
            {
  "where": {
    "field": "camera_zone",
    "op": "in",
    "value": [
      "north",
      "east"
    ]
  }
}
            ```


In [18]:
case_result = run_filter_case(SESSION, 'op_in')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_in',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 0.7388502579584794,
   'metadata': {'tags': ['traffic', 'vehicle', 'red'],
    'prefix': 'alpha-route',
    'video_id': 'vid-001',
    'created_at': '2026-03-10T10:00:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'frame_number': 10,
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 0.5779719226947998,
   'metadata': {'tags': ['pedestrian', 'bicycle'],
    'prefix': 'alpha-bike',
    'video_id': 'vid-003',
    'created_at': '2026-03-20T08:30:00+00:00',
    'bucket_name': 'cam-c',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'frame_number': 30,
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'field'

## Filter case: `op_contains`

            **Family:** String containment

            This cell mirrors the `op_contains` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-001`, `vid-006`.

            ```json
            {
  "where": {
    "field": "description",
    "op": "contains",
    "value": "intersection"
  }
}
            ```


In [19]:
case_result = run_filter_case(SESSION, 'op_contains')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_contains',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 0.7388502579584794,
   'metadata': {'tags': ['traffic', 'vehicle', 'red'],
    'prefix': 'alpha-route',
    'video_id': 'vid-001',
    'created_at': '2026-03-10T10:00:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'frame_number': 10,
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 0.6368856321837195,
   'metadata': {'tags': ['night', 'traffic'],
    'prefix': 'delta-night',
    'video_id': 'vid-006',
    'created_at': '2026-04-05T20:10:00+00:00',
    'bucket_name': 'cam-e',
    'camera_zone': 'south-west',
    'description': 'empty intersection at night',
    'frame_number': 60},
   'page_content': 'empty intersection at night'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'field': 'description',
   'op': 'contains

## Filter case: `op_starts_with`

            **Family:** String prefix

            This cell mirrors the `op_starts_with` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-001`, `vid-003`, `vid-005`.

            ```json
            {
  "where": {
    "field": "prefix",
    "op": "starts_with",
    "value": "alpha"
  }
}
            ```


In [20]:
case_result = run_filter_case(SESSION, 'op_starts_with')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_starts_with',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 0.7388502579584794,
   'metadata': {'tags': ['traffic', 'vehicle', 'red'],
    'prefix': 'alpha-route',
    'video_id': 'vid-001',
    'created_at': '2026-03-10T10:00:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'frame_number': 10,
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 0.5779719226947998,
   'metadata': {'tags': ['pedestrian', 'bicycle'],
    'prefix': 'alpha-bike',
    'video_id': 'vid-003',
    'created_at': '2026-03-20T08:30:00+00:00',
    'bucket_name': 'cam-c',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'frame_number': 30,
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.46225378844943255,
   'metadata': {'tags': ['traffic', 'signal'],
    'prefix': 'alph

## Filter case: `op_gt`

            **Family:** Numeric comparison

            This cell mirrors the `op_gt` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-005`, `vid-006`.

            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "gt",
    "value": 40
  }
}
            ```


In [21]:
case_result = run_filter_case(SESSION, 'op_gt')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_gt',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 0.6368856321837195,
   'metadata': {'tags': ['night', 'traffic'],
    'prefix': 'delta-night',
    'video_id': 'vid-006',
    'created_at': '2026-04-05T20:10:00+00:00',
    'bucket_name': 'cam-e',
    'camera_zone': 'south-west',
    'description': 'empty intersection at night',
    'frame_number': 60},
   'page_content': 'empty intersection at night'},
  {'score': 0.46225378844943255,
   'metadata': {'tags': ['traffic', 'signal'],
    'prefix': 'alpha-signal',
    'video_id': 'vid-005',
    'created_at': '2026-04-01T09:45:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north-east',
    'description': 'traffic light malfunction',
    'frame_number': 50,
    'optional_note': 'signal issue'},
   'page_content': 'traffic light malfunction'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'field': 'frame_number',
   'op': 'gt',
   'v

## Filter case: `op_gte`

            **Family:** Numeric comparison

            This cell mirrors the `op_gte` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-004`, `vid-005`, `vid-006`.

            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "gte",
    "value": 40
  }
}
            ```


In [22]:
case_result = run_filter_case(SESSION, 'op_gte')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_gte',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 0.6368856321837195,
   'metadata': {'tags': ['night', 'traffic'],
    'prefix': 'delta-night',
    'video_id': 'vid-006',
    'created_at': '2026-04-05T20:10:00+00:00',
    'bucket_name': 'cam-e',
    'camera_zone': 'south-west',
    'description': 'empty intersection at night',
    'frame_number': 60},
   'page_content': 'empty intersection at night'},
  {'score': 0.46225378844943255,
   'metadata': {'tags': ['traffic', 'signal'],
    'prefix': 'alpha-signal',
    'video_id': 'vid-005',
    'created_at': '2026-04-01T09:45:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north-east',
    'description': 'traffic light malfunction',
    'frame_number': 50,
    'optional_note': 'signal issue'},
   'page_content': 'traffic light malfunction'},
  {'score': 0.45352051459558307,
   'metadata': {'tags': ['logistics', 'vehicle'],
    'prefix': 'gamma-log',
    'video_id': 'vid-004',
    'cre

## Filter case: `op_lt`

            **Family:** Numeric comparison

            This cell mirrors the `op_lt` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-001`, `vid-002`.

            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "lt",
    "value": 30
  }
}
            ```


In [23]:
case_result = run_filter_case(SESSION, 'op_lt')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_lt',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 0.7388502579584794,
   'metadata': {'tags': ['traffic', 'vehicle', 'red'],
    'prefix': 'alpha-route',
    'video_id': 'vid-001',
    'created_at': '2026-03-10T10:00:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'frame_number': 10,
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 0.735431216222978,
   'metadata': {'tags': ['traffic', 'vehicle', 'bus'],
    'prefix': 'beta-route',
    'video_id': 'vid-002',
    'created_at': '2026-03-15T12:00:00+00:00',
    'bucket_name': 'cam-b',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'frame_number': 20},
   'page_content': 'blue bus near station'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'field': 'frame_number',
   'op': 'lt',
   'value': 30,
  

## Filter case: `op_lte`

            **Family:** Numeric comparison

            This cell mirrors the `op_lte` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-001`, `vid-002`, `vid-003`.

            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "lte",
    "value": 30
  }
}
            ```


In [24]:
case_result = run_filter_case(SESSION, 'op_lte')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_lte',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 0.7388502579584794,
   'metadata': {'tags': ['traffic', 'vehicle', 'red'],
    'prefix': 'alpha-route',
    'video_id': 'vid-001',
    'created_at': '2026-03-10T10:00:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'frame_number': 10,
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 0.735431216222978,
   'metadata': {'tags': ['traffic', 'vehicle', 'bus'],
    'prefix': 'beta-route',
    'video_id': 'vid-002',
    'created_at': '2026-03-15T12:00:00+00:00',
    'bucket_name': 'cam-b',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'frame_number': 20},
   'page_content': 'blue bus near station'},
  {'score': 0.5779719226947998,
   'metadata': {'tags': ['pedestrian', 'bicycle'],
    'prefix': 'alpha-bike',
    'video_id': 'vid-003',
    'created_at': '2

## Filter case: `op_between`

            **Family:** Numeric range

            This cell mirrors the `op_between` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-002`, `vid-003`, `vid-004`.

            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "between",
    "value": [
      20,
      40
    ]
  }
}
            ```


In [25]:
case_result = run_filter_case(SESSION, 'op_between')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_between',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 0.735431216222978,
   'metadata': {'tags': ['traffic', 'vehicle', 'bus'],
    'prefix': 'beta-route',
    'video_id': 'vid-002',
    'created_at': '2026-03-15T12:00:00+00:00',
    'bucket_name': 'cam-b',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'frame_number': 20},
   'page_content': 'blue bus near station'},
  {'score': 0.5779719226947998,
   'metadata': {'tags': ['pedestrian', 'bicycle'],
    'prefix': 'alpha-bike',
    'video_id': 'vid-003',
    'created_at': '2026-03-20T08:30:00+00:00',
    'bucket_name': 'cam-c',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'frame_number': 30,
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.45352051459558307,
   'metadata': {'tags': ['logistics', 'vehicle'],
    'prefix': 'gamma-log',
    'video_id': 'vid-004',
    'crea

## Filter case: `op_contains_any`

            **Family:** Array membership

            This cell mirrors the `op_contains_any` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-003`, `vid-005`.

            ```json
            {
  "where": {
    "field": "tags",
    "op": "contains_any",
    "value": [
      "bicycle",
      "signal"
    ]
  }
}
            ```


In [26]:
case_result = run_filter_case(SESSION, 'op_contains_any')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_contains_any',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 0.5779719226947998,
   'metadata': {'tags': ['pedestrian', 'bicycle'],
    'prefix': 'alpha-bike',
    'video_id': 'vid-003',
    'created_at': '2026-03-20T08:30:00+00:00',
    'bucket_name': 'cam-c',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'frame_number': 30,
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.46225378844943255,
   'metadata': {'tags': ['traffic', 'signal'],
    'prefix': 'alpha-signal',
    'video_id': 'vid-005',
    'created_at': '2026-04-01T09:45:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north-east',
    'description': 'traffic light malfunction',
    'frame_number': 50,
    'optional_note': 'signal issue'},
   'page_content': 'traffic light malfunction'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_whe

## Filter case: `op_contains_all`

            **Family:** Array membership

            This cell mirrors the `op_contains_all` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-001`, `vid-002`.

            ```json
            {
  "where": {
    "field": "tags",
    "op": "contains_all",
    "value": [
      "traffic",
      "vehicle"
    ]
  }
}
            ```


In [27]:
case_result = run_filter_case(SESSION, 'op_contains_all')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_contains_all',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 0.7388502579584794,
   'metadata': {'tags': ['traffic', 'vehicle', 'red'],
    'prefix': 'alpha-route',
    'video_id': 'vid-001',
    'created_at': '2026-03-10T10:00:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'frame_number': 10,
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 0.735431216222978,
   'metadata': {'tags': ['traffic', 'vehicle', 'bus'],
    'prefix': 'beta-route',
    'video_id': 'vid-002',
    'created_at': '2026-03-15T12:00:00+00:00',
    'bucket_name': 'cam-b',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'frame_number': 20},
   'page_content': 'blue bus near station'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'field': 'tags',
   'op': 'contains_all',
   'va

## Filter case: `op_exists`

            **Family:** Field presence

            This cell mirrors the `op_exists` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-001`, `vid-003`, `vid-005`.

            ```json
            {
  "where": {
    "field": "optional_note",
    "op": "exists"
  }
}
            ```


In [28]:
case_result = run_filter_case(SESSION, 'op_exists')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_exists',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 0.7388502579584794,
   'metadata': {'tags': ['traffic', 'vehicle', 'red'],
    'prefix': 'alpha-route',
    'video_id': 'vid-001',
    'created_at': '2026-03-10T10:00:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'frame_number': 10,
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 0.5779719226947998,
   'metadata': {'tags': ['pedestrian', 'bicycle'],
    'prefix': 'alpha-bike',
    'video_id': 'vid-003',
    'created_at': '2026-03-20T08:30:00+00:00',
    'bucket_name': 'cam-c',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'frame_number': 30,
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.46225378844943255,
   'metadata': {'tags': ['traffic', 'signal'],
    'prefix': 'alpha-sig

## Filter case: `op_missing`

            **Family:** Field absence

            This cell mirrors the `op_missing` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-002`, `vid-004`, `vid-006`.

            ```json
            {
  "where": {
    "field": "optional_note",
    "op": "missing"
  }
}
            ```


In [29]:
case_result = run_filter_case(SESSION, 'op_missing')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_missing',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 0.735431216222978,
   'metadata': {'tags': ['traffic', 'vehicle', 'bus'],
    'prefix': 'beta-route',
    'video_id': 'vid-002',
    'created_at': '2026-03-15T12:00:00+00:00',
    'bucket_name': 'cam-b',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'frame_number': 20},
   'page_content': 'blue bus near station'},
  {'score': 0.6368856321837195,
   'metadata': {'tags': ['night', 'traffic'],
    'prefix': 'delta-night',
    'video_id': 'vid-006',
    'created_at': '2026-04-05T20:10:00+00:00',
    'bucket_name': 'cam-e',
    'camera_zone': 'south-west',
    'description': 'empty intersection at night',
    'frame_number': 60},
   'page_content': 'empty intersection at night'},
  {'score': 0.45352051459558307,
   'metadata': {'tags': ['logistics', 'vehicle'],
    'prefix': 'gamma-log',
    'video_id': 'vid-004',
    'created_at': '2026-03-25T14:15:00+00:00',
   

## Filter case: `logical_compound`

            **Family:** Logical composition

            This cell mirrors the `logical_compound` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-003`, `vid-004`.

            ```json
            {
  "where": {
    "all": [
      {
        "field": "frame_number",
        "op": "gte",
        "value": 20
      },
      {
        "any": [
          {
            "field": "camera_zone",
            "op": "eq",
            "value": "east"
          },
          {
            "field": "camera_zone",
            "op": "eq",
            "value": "west"
          }
        ]
      },
      {
        "not": {
          "field": "tags",
          "op": "contains_any",
          "value": [
            "night"
          ]
        }
      }
    ]
  }
}
            ```


In [30]:
case_result = run_filter_case(SESSION, 'logical_compound')
case_result


<IPython.core.display.JSON object>

{'query_id': 'logical_compound',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 0.5779719226947998,
   'metadata': {'tags': ['pedestrian', 'bicycle'],
    'prefix': 'alpha-bike',
    'video_id': 'vid-003',
    'created_at': '2026-03-20T08:30:00+00:00',
    'bucket_name': 'cam-c',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'frame_number': 30,
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.45352051459558307,
   'metadata': {'tags': ['logistics', 'vehicle'],
    'prefix': 'gamma-log',
    'video_id': 'vid-004',
    'created_at': '2026-03-25T14:15:00+00:00',
    'bucket_name': 'cam-d',
    'camera_zone': 'west',
    'description': 'delivery truck unloading',
    'frame_number': 40},
   'page_content': 'delivery truck unloading'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'field': None,
   'op': None,
   'valu

## Filter case: `legacy_tags_and_time_filter`

            **Family:** Legacy compatibility

            This cell mirrors the `legacy_tags_and_time_filter` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-002`, `vid-005`.

            ```json
            {
  "tags": [
    "traffic"
  ],
  "time_filter": {
    "end": "2026-04-02T00:00:00+00:00",
    "start": "2026-03-12T00:00:00+00:00"
  }
}
            ```


In [31]:
case_result = run_filter_case(SESSION, 'legacy_tags_and_time_filter')
case_result


<IPython.core.display.JSON object>

{'query_id': 'legacy_tags_and_time_filter',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 0.735431216222978,
   'metadata': {'tags': ['traffic', 'vehicle', 'bus'],
    'prefix': 'beta-route',
    'video_id': 'vid-002',
    'created_at': '2026-03-15T12:00:00+00:00',
    'bucket_name': 'cam-b',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'frame_number': 20},
   'page_content': 'blue bus near station'},
  {'score': 0.46225378844943255,
   'metadata': {'tags': ['traffic', 'signal'],
    'prefix': 'alpha-signal',
    'video_id': 'vid-005',
    'created_at': '2026-04-01T09:45:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north-east',
    'description': 'traffic light malfunction',
    'frame_number': 50,
    'optional_note': 'signal issue'},
   'page_content': 'traffic light malfunction'}],
 'applied_filters': {'tags': ['traffic'],
  'time_filter': {'start': '2026-03-12T00:00:00Z',
   'end': '2026-04-02T00:00:00Z'},
  'fi

## Filter case: `legacy_filters_map`

            **Family:** Legacy compatibility

            This cell mirrors the `legacy_filters_map` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-001`, `vid-002`.

            ```json
            {
  "filters": {
    "bucket_name": {
      "op": "in",
      "value": [
        "cam-a",
        "cam-b"
      ]
    },
    "frame_number": {
      "op": "between",
      "value": [
        10,
        20
      ]
    }
  }
}
            ```


In [32]:
case_result = run_filter_case(SESSION, 'legacy_filters_map')
case_result


<IPython.core.display.JSON object>

{'query_id': 'legacy_filters_map',
 'query': 'fixture retrieval anchor',
 'count': 2,
 'items': [{'score': 0.7388502579584794,
   'metadata': {'tags': ['traffic', 'vehicle', 'red'],
    'prefix': 'alpha-route',
    'video_id': 'vid-001',
    'created_at': '2026-03-10T10:00:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'frame_number': 10,
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 0.735431216222978,
   'metadata': {'tags': ['traffic', 'vehicle', 'bus'],
    'prefix': 'beta-route',
    'video_id': 'vid-002',
    'created_at': '2026-03-15T12:00:00+00:00',
    'bucket_name': 'cam-b',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'frame_number': 20},
   'page_content': 'blue bus near station'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': {'bucket_name': {'op': 'in', 'value': ['cam-a', 'cam-b']},
   'frame_numbe

## Filter case: `logical_not_toplevel`

            **Family:** Logical composition

            This cell mirrors the `logical_not_toplevel` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-002`, `vid-003`, `vid-004`, `vid-005`, `vid-006`.

            ```json
            {
  "where": {
    "not": {
      "field": "camera_zone",
      "op": "eq",
      "value": "north"
    }
  }
}
            ```


In [33]:
case_result = run_filter_case(SESSION, 'logical_not_toplevel')
case_result


<IPython.core.display.JSON object>

{'query_id': 'logical_not_toplevel',
 'query': 'fixture retrieval anchor',
 'count': 5,
 'items': [{'score': 0.735431216222978,
   'metadata': {'tags': ['traffic', 'vehicle', 'bus'],
    'prefix': 'beta-route',
    'video_id': 'vid-002',
    'created_at': '2026-03-15T12:00:00+00:00',
    'bucket_name': 'cam-b',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'frame_number': 20},
   'page_content': 'blue bus near station'},
  {'score': 0.6368856321837195,
   'metadata': {'tags': ['night', 'traffic'],
    'prefix': 'delta-night',
    'video_id': 'vid-006',
    'created_at': '2026-04-05T20:10:00+00:00',
    'bucket_name': 'cam-e',
    'camera_zone': 'south-west',
    'description': 'empty intersection at night',
    'frame_number': 60},
   'page_content': 'empty intersection at night'},
  {'score': 0.5779719226947998,
   'metadata': {'tags': ['pedestrian', 'bicycle'],
    'prefix': 'alpha-bike',
    'video_id': 'vid-003',
    'created_at': '2026-03-20T08:30:00+

## Filter case: `no_match`

            **Family:** Zero-result behavior

            This cell mirrors the `no_match` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: _no matching documents_.

            ```json
            {
  "where": {
    "field": "video_id",
    "op": "eq",
    "value": "vid-999"
  }
}
            ```


In [34]:
case_result = run_filter_case(SESSION, 'no_match')
case_result


<IPython.core.display.JSON object>

{'query_id': 'no_match',
 'query': 'fixture retrieval anchor',
 'count': 0,
 'items': [],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'field': 'video_id',
   'op': 'eq',
   'value': 'vid-999',
   'all': None,
   'any': None,
   'not': None},
  'warnings': None,
  'compiled_backend_filter': {'video_id': {'$eq': 'vid-999'}},
  'dropped_or_rewritten_clauses': []}}

## Filter case: `op_time_between_where`

            **Family:** Datetime range

            This cell mirrors the `op_time_between_where` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-003`, `vid-004`, `vid-005`.

            ```json
            {
  "where": {
    "field": "created_at",
    "op": "between",
    "value": [
      "2026-03-20T00:00:00+00:00",
      "2026-04-01T23:59:59+00:00"
    ]
  }
}
            ```


In [35]:
case_result = run_filter_case(SESSION, 'op_time_between_where')
case_result


<IPython.core.display.JSON object>

{'query_id': 'op_time_between_where',
 'query': 'fixture retrieval anchor',
 'count': 3,
 'items': [{'score': 0.5779719226947998,
   'metadata': {'tags': ['pedestrian', 'bicycle'],
    'prefix': 'alpha-bike',
    'video_id': 'vid-003',
    'created_at': '2026-03-20T08:30:00+00:00',
    'bucket_name': 'cam-c',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'frame_number': 30,
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'},
  {'score': 0.46225378844943255,
   'metadata': {'tags': ['traffic', 'signal'],
    'prefix': 'alpha-signal',
    'video_id': 'vid-005',
    'created_at': '2026-04-01T09:45:00+00:00',
    'bucket_name': 'cam-a',
    'camera_zone': 'north-east',
    'description': 'traffic light malfunction',
    'frame_number': 50,
    'optional_note': 'signal issue'},
   'page_content': 'traffic light malfunction'},
  {'score': 0.45352051459558307,
   'metadata': {'tags': ['logistics', 'vehicle'],
    'p

## Filter case: `nested_all`

            **Family:** Nested logical composition

            This cell mirrors the `nested_all` entry from `tests.functional.data.FILTER_CASES`.
            Expected matching `video_id` values: `vid-003`.

            ```json
            {
  "where": {
    "all": [
      {
        "all": [
          {
            "field": "frame_number",
            "op": "gte",
            "value": 10
          },
          {
            "field": "frame_number",
            "op": "lte",
            "value": 30
          }
        ]
      },
      {
        "field": "bucket_name",
        "op": "eq",
        "value": "cam-c"
      }
    ]
  }
}
            ```


In [36]:
case_result = run_filter_case(SESSION, 'nested_all')
case_result


<IPython.core.display.JSON object>

{'query_id': 'nested_all',
 'query': 'fixture retrieval anchor',
 'count': 1,
 'items': [{'score': 0.5779719226947998,
   'metadata': {'tags': ['pedestrian', 'bicycle'],
    'prefix': 'alpha-bike',
    'video_id': 'vid-003',
    'created_at': '2026-03-20T08:30:00+00:00',
    'bucket_name': 'cam-c',
    'camera_zone': 'east',
    'description': 'person with bicycle crossing',
    'frame_number': 30,
    'optional_note': 'pedestrian event'},
   'page_content': 'person with bicycle crossing'}],
 'applied_filters': {'tags': None,
  'time_filter': None,
  'filters': None,
  'normalized_where': {'field': None,
   'op': None,
   'value': None,
   'all': [{'field': None,
     'op': None,
     'value': None,
     'all': [{'field': 'frame_number',
       'op': 'gte',
       'value': 10,
       'all': None,
       'any': None,
       'not': None},
      {'field': 'frame_number',
       'op': 'lte',
       'value': 30,
       'all': None,
       'any': None,
       'not': None}],
     'any': None,

## Cleanup checkpoint

Before tearing the stack down, you can inspect the current Docker Compose view of the
PGVector walkthrough services.


In [37]:
print(compose_ps(SESSION))


$ docker compose -f docker/compose.yaml -f docker/compose.pgvector.yaml ps
NAME                           IMAGE                                       COMMAND                  SERVICE                        CREATED          STATUS                    PORTS
docker-milvus-etcd-1           quay.io/coreos/etcd:v3.5.18                 "etcd -advertise-cli…"   milvus-etcd                    5 minutes ago    Up 5 minutes (healthy)    2379-2380/tcp
docker-milvus-standalone-1     milvusdb/milvus:v2.6.14                     "/tini -- milvus run…"   milvus-standalone              5 minutes ago    Up 4 minutes (healthy)    0.0.0.0:9092->9091/tcp, [::]:9092->9091/tcp, 0.0.0.0:19531->19530/tcp, [::]:19531->19530/tcp
docker-pgvector-db-1           pgvector/pgvector:pg18                      "docker-entrypoint.s…"   pgvector-db                    14 seconds ago   Up 12 seconds (healthy)   0.0.0.0:5433->5432/tcp, [::]:5433->5432/tcp
docker-vector-retriever-1      vector-retriever-pgvector:latest         

## Teardown

Run this cell when you are done exploring. It stops the compose stack and removes the
associated volumes created for the walkthrough.


In [38]:
teardown_output = teardown_stack(SESSION)
print(teardown_output)


$ docker compose -f docker/compose.yaml -f docker/compose.pgvector.yaml down -v --remove-orphans
time="2026-05-25T10:50:34+05:30" level=warning msg="The \"REGISTRY\" variable is not set. Defaulting to a blank string."
 Container docker-vector-retriever-1 Stopping 
 Container docker-vector-retriever-1 Stopped 
 Container docker-vector-retriever-1 Removing 
 Container docker-vector-retriever-1 Removed 
 Container multimodal-embedding-serving Stopping 
 Container docker-pgvector-db-1 Stopping 
 Container docker-pgvector-db-1 Stopped 
 Container docker-pgvector-db-1 Removing 
 Container docker-pgvector-db-1 Removed 
 Container multimodal-embedding-serving Stopped 
 Container multimodal-embedding-serving Removing 
 Container multimodal-embedding-serving Removed 
 Container docker-milvus-etcd-1 Stopping 
 Container docker-milvus-standalone-1 Stopping 
 Container docker-milvus-standalone-1 Stopped 
 Container docker-milvus-standalone-1 Removing 
 Container docker-milvus-etcd-1 Stopped 
 Conta